In [10]:
# Import libraries
from dataclasses import dataclass
from typing import List
from transformers import pipeline

In [11]:
# Cell 2: Load the AI models (this might take a minute)
print("Loading AI models...")

# Sentiment model (positive/negative)
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Emotion detection model (happy, sad, anger, etc.)
emotion_model = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base"
)

print("✅ Models loaded successfully!")

Loading AI models...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Models loaded successfully!


In [12]:
# Cell 3: Simple data class to store user input
@dataclass
class WellnessInput:
    mood: int          # Rate 1-5 (1=very bad, 5=very good)
    stress: int        # Rate 1-10 (1=no stress, 10=extreme stress)
    sleep: int         # Hours of sleep last night
    energy: int        # Rate 1-10 (1=no energy, 10=full energy)
    productivity: int  # Rate 1-10 (1=not productive, 10=very productive)
    message: str       # How are you feeling today? (write a sentence)

In [13]:
# Cell 4: Simple function to calculate overall wellness score
def calculate_wellness_score(data):
    # Convert all inputs to 0-100 scale
    mood_score = data.mood * 20                    # 1-5 → 20-100
    stress_score = (10 - data.stress) * 10         # 1-10 → 90-0 (inverted)
    sleep_score = min(data.sleep * 10, 100)        # hours → max 100
    energy_score = data.energy * 10                 # 1-10 → 10-100
    productivity_score = data.productivity * 10     # 1-10 → 10-100
    
    # Calculate average
    total = (mood_score + stress_score + sleep_score + energy_score + productivity_score)
    average = total / 5
    
    return round(average, 1)

In [14]:
# Cell 5: Simple burnout risk assessment
def check_burnout_risk(data):
    risk_points = 0
    
    # Add points for each risk factor
    if data.stress > 7:
        risk_points += 2
    if data.sleep < 6:
        risk_points += 2
    if data.energy < 4:
        risk_points += 1
    if data.productivity < 4:
        risk_points += 1
    if data.mood < 3:
        risk_points += 1
    
    # Determine risk level
    if risk_points >= 5:
        return "🔴 HIGH RISK - Please take care of yourself!"
    elif risk_points >= 3:
        return "🟡 MEDIUM RISK - Consider making some changes"
    else:
        return "🟢 LOW RISK - You're doing great!"

In [15]:
# Cell 6: Get personalized recommendations
def get_recommendations(data):
    recommendations = []
    
    # Analyze message with AI
    sentiment = sentiment_model(data.message)[0]
    emotion = emotion_model(data.message)[0]
    
    # Stress-based recommendations
    if data.stress >= 8:
        recommendations.append("🧘 Take a 10-minute break to breathe and relax")
    elif data.stress >= 5:
        recommendations.append("🎵 Listen to calming music for 15 minutes")
    
    # Sleep-based recommendations
    if data.sleep < 6:
        recommendations.append("😴 Try to sleep earlier tonight - aim for 7-8 hours")
    
    # Energy-based recommendations
    if data.energy < 5:
        recommendations.append("🚶 Go for a short 10-minute walk")
    
    # Mood-based recommendations  
    if data.mood <= 2:
        recommendations.append("💬 Talk to a friend or family member")
    
    # AI sentiment-based
    if sentiment['label'] == 'NEGATIVE':
        recommendations.append("📝 Write down three things you're grateful for")
    
    # AI emotion-based
    if emotion['label'] == 'sadness':
        recommendations.append("🎬 Watch something funny or uplifting")
    elif emotion['label'] == 'anger':
        recommendations.append("💪 Try some physical exercise to release tension")
    elif emotion['label'] == 'joy':
        recommendations.append("✨ Keep sharing your positive energy with others!")
    
    # Add general recommendation if none specific
    if len(recommendations) == 0:
        recommendations.append("🌟 You're doing well! Keep maintaining your healthy habits")
    
    return recommendations, sentiment, emotion

In [16]:
# Cell 7: Main function that puts it all together
def wellness_coach(data):
    # Calculate all metrics
    score = calculate_wellness_score(data)
    risk = check_burnout_risk(data)
    recommendations, sentiment, emotion = get_recommendations(data)
    
    # Return all results
    return {
        "wellness_score": score,
        "burnout_risk": risk,
        "sentiment": sentiment,
        "emotion": emotion['label'],
        "recommendations": recommendations
    }

In [17]:
# Cell 8: Test the wellness coach with sample data

# Create a test input
test_data = WellnessInput(
    mood=2,           # Feeling bad
    stress=8,         # Very stressed
    sleep=5,          # Only 5 hours sleep
    energy=3,         # Low energy
    productivity=4,   # Not very productive
    message="I'm really stressed with work and feeling exhausted"  # How you feel
)

# Get wellness recommendation
result = wellness_coach(test_data)

# Print results nicely
print("=" * 50)
print("🤖 AI WELLNESS COACH REPORT")
print("=" * 50)
print(f"📊 Wellness Score: {result['wellness_score']}%")
print(f"⚠️  Burnout Risk: {result['burnout_risk']}")
print(f"😊 Detected Emotion: {result['emotion']}")
print(f"📝 Message Sentiment: {result['sentiment']['label']} ({result['sentiment']['score']:.0%} confident)")
print("\n💡 Your Personalized Recommendations:")
print("-" * 40)
for i, rec in enumerate(result['recommendations'], 1):
    print(f"  {i}. {rec}")
print("=" * 50)

🤖 AI WELLNESS COACH REPORT
📊 Wellness Score: 36.0%
⚠️  Burnout Risk: 🔴 HIGH RISK - Please take care of yourself!
😊 Detected Emotion: sadness
📝 Message Sentiment: NEGATIVE (100% confident)

💡 Your Personalized Recommendations:
----------------------------------------
  1. 🧘 Take a 10-minute break to breathe and relax
  2. 😴 Try to sleep earlier tonight - aim for 7-8 hours
  3. 🚶 Go for a short 10-minute walk
  4. 💬 Talk to a friend or family member
  5. 📝 Write down three things you're grateful for
  6. 🎬 Watch something funny or uplifting
